# Olist E-Commerce Analysis

## Module 2 — Data Engineering and Business Analysis

This notebook presents the analytical layer of the Olist data pipeline.

The warehouse is hosted in **Google BigQuery** and is accessed from Python using **SQLAlchemy** and **Pandas**. Reusable analytical queries are maintained in `queries.py`, while `analysis.py` exports the standard analysis outputs to CSV and `visualizations.py` creates presentation-ready PNG charts.

### Analysis scope

The notebook covers:

1. Monthly sales trend by customer region
2. November 2017 sales-spike investigation
3. RFM customer analysis
4. Holiday vs non-holiday analysis
5. Holiday product-category mix
6. Key business findings

### Analysis periods

- **Regional monthly sales trend:** Jan 2017 – Jun 2018
- **November spike investigation:** Oct–Dec 2017, with daily drill-down for Nov 2017
- **RFM analysis:** full available RFM mart
- **Holiday analysis:** Jan 2017 – Jun 2018

> The 2016 observations are excluded from the regional trend because the year is incomplete. The regional trend also stops at Jun 2018 to keep the comparison period consistent.

## 1. Setup

Before opening the notebook, the standard project workflow is:

```bash
python analysis.py
python check_csvs.py
python visualizations.py
jupyter notebook
```

`analysis.py` queries BigQuery and exports CSV results.  
`check_csvs.py` validates the generated outputs.  
`visualizations.py` produces the PNG charts used throughout this notebook.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image, display
from sqlalchemy import text

from config import PROJECT_ID, DATASET, OUTPUT_DIR
from engine import engine, test_connection, verify_required_tables

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Project:", PROJECT_ID)
print("Dataset:", DATASET)
print("Output directory:", OUTPUT_DIR)

### Validate BigQuery connection and mart tables

In [ ]:
test_connection(engine)
verify_required_tables(engine)

print("BigQuery connection and required mart-table checks completed.")

## 2. Monthly Sales Trend by Customer Region

The monthly sales analysis groups Brazilian states into the five geographic regions:

- Southeast
- South
- Northeast
- Central-West
- North

The metric plotted is **GMV = item price + freight value**.

The analysis period is restricted to **Jan 2017 – Jun 2018** to avoid the incomplete 2016 period and later partial data.

In [ ]:
monthly_region = pd.read_csv(
    OUTPUT_DIR / "monthly_sales_trend.csv"
)

monthly_region.head(10)

In [ ]:
display(
    Image(
        filename=str(
            OUTPUT_DIR / "monthly_sales_trend.png"
        )
    )
)

### Regional trend observations

The Southeast is the dominant sales region throughout the analysis period. South and Northeast form the second tier, while Central-West and North contribute smaller monthly GMV.

A notable feature is the **November 2017 spike across all regions**. Because the increase appears simultaneously across the country, the next section investigates whether the spike was driven by:

- more orders,
- higher average order value,
- a small number of high-activity days, or
- specific product categories.

## 3. November 2017 Spike Investigation

### 3.1 Order volume vs Average Order Value

This query compares Oct, Nov and Dec 2017. It helps distinguish a **volume-driven** sales increase from an **AOV-driven** increase.

In [ ]:
order_vs_aov_sql = f"""
WITH order_totals AS (
    SELECT
        order_key,
        SUM(price + freight_value) AS order_value
    FROM `{PROJECT_ID}.{DATASET}.fact_order_items`
    GROUP BY order_key
)

SELECT
    FORMAT('%04d-%02d', d.year, d.month) AS year_month,
    COUNT(DISTINCT f.order_id) AS orders,
    ROUND(SUM(ot.order_value), 2) AS gmv,
    ROUND(
        SAFE_DIVIDE(
            SUM(ot.order_value),
            COUNT(DISTINCT f.order_id)
        ),
        2
    ) AS avg_order_value

FROM `{PROJECT_ID}.{DATASET}.fact_orders` AS f

JOIN `{PROJECT_ID}.{DATASET}.dim_date` AS d
    ON f.order_date_key = d.date_key

LEFT JOIN order_totals AS ot
    ON f.order_key = ot.order_key

WHERE d.year = 2017
  AND d.month IN (10, 11, 12)

GROUP BY
    d.year,
    d.month

ORDER BY
    d.year,
    d.month
"""

order_vs_aov = pd.read_sql(
    text(order_vs_aov_sql),
    con=engine,
)

order_vs_aov

In [ ]:
oct_row = order_vs_aov.loc[
    order_vs_aov["year_month"] == "2017-10"
].iloc[0]

nov_row = order_vs_aov.loc[
    order_vs_aov["year_month"] == "2017-11"
].iloc[0]

dec_row = order_vs_aov.loc[
    order_vs_aov["year_month"] == "2017-12"
].iloc[0]

oct_to_nov_orders_pct = (
    (nov_row["orders"] / oct_row["orders"]) - 1
) * 100

oct_to_nov_gmv_pct = (
    (nov_row["gmv"] / oct_row["gmv"]) - 1
) * 100

oct_to_nov_aov_pct = (
    (nov_row["avg_order_value"] / oct_row["avg_order_value"]) - 1
) * 100

print(f"Oct → Nov order growth: {oct_to_nov_orders_pct:.1f}%")
print(f"Oct → Nov GMV growth:   {oct_to_nov_gmv_pct:.1f}%")
print(f"Oct → Nov AOV change:   {oct_to_nov_aov_pct:.1f}%")

### Interpretation

The observed results show that November 2017 was primarily **volume-driven**:

- Orders increased sharply from October to November.
- GMV also increased strongly.
- Average Order Value decreased rather than increased.

Therefore, the November spike was caused mainly by **more orders being placed**, rather than customers spending more per transaction.

### 3.2 Daily activity during November 2017

In [ ]:
nov_daily_sql = f"""
SELECT
    d.date_key,
    COUNT(DISTINCT i.order_key) AS order_count,
    ROUND(
        SUM(i.price + i.freight_value),
        2
    ) AS gmv

FROM `{PROJECT_ID}.{DATASET}.fact_order_items` AS i

JOIN `{PROJECT_ID}.{DATASET}.dim_date` AS d
    ON i.order_date_key = d.date_key

WHERE d.year = 2017
  AND d.month = 11

GROUP BY d.date_key

ORDER BY d.date_key
"""

nov_daily = pd.read_sql(
    text(nov_daily_sql),
    con=engine,
)

nov_daily.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(nov_daily))

ax.plot(
    x,
    nov_daily["gmv"],
    marker="o",
)

ax.set_xticks(list(x))
ax.set_xticklabels(
    nov_daily["date_key"].astype(str),
    rotation=90,
)

ax.set_title(
    "Daily GMV — November 2017"
)

ax.set_xlabel("Date")
ax.set_ylabel("GMV")
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
peak = nov_daily.loc[
    nov_daily["gmv"].idxmax()
]

print("Peak date:", int(peak["date_key"]))
print("Peak orders:", int(peak["order_count"]))
print("Peak GMV:", f'{peak["gmv"]:,.2f}')

### Daily-spike observation

The daily drill-down identifies **24 Nov 2017** as the dominant peak which is **BLACK FRIDAY**. The increase is therefore not spread evenly throughout November; a substantial part of the monthly increase is concentrated around a short event period.

This pattern is consistent with an event-driven shopping surge. The analysis demonstrates the event pattern from the transaction data; causal attribution should be stated carefully unless external promotional data is also introduced.

### 3.3 Product categories responsible for the October → November increase

In [ ]:
category_spike_sql = f"""
WITH category_monthly AS (

    SELECT
        d.month,

        COALESCE(
            p.product_category_name_english,
            'Unknown'
        ) AS product_category,

        COUNT(*) AS units_sold,

        COUNT(
            DISTINCT i.order_key
        ) AS order_count,

        SUM(
            i.price + i.freight_value
        ) AS gmv

    FROM `{PROJECT_ID}.{DATASET}.fact_order_items` AS i

    JOIN `{PROJECT_ID}.{DATASET}.dim_date` AS d
        ON i.order_date_key = d.date_key

    LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_product` AS p
        ON i.product_key = p.product_key

    WHERE d.year = 2017
      AND d.month IN (10, 11)

    GROUP BY
        d.month,
        product_category
),

comparison AS (

    SELECT
        product_category,

        SUM(
            CASE WHEN month = 10
                 THEN gmv ELSE 0 END
        ) AS oct_gmv,

        SUM(
            CASE WHEN month = 11
                 THEN gmv ELSE 0 END
        ) AS nov_gmv,

        SUM(
            CASE WHEN month = 10
                 THEN order_count ELSE 0 END
        ) AS oct_orders,

        SUM(
            CASE WHEN month = 11
                 THEN order_count ELSE 0 END
        ) AS nov_orders

    FROM category_monthly

    GROUP BY product_category
),

uplift AS (

    SELECT
        *,
        nov_gmv - oct_gmv AS gmv_increase,

        SAFE_MULTIPLY(
            SAFE_DIVIDE(
                nov_gmv - oct_gmv,
                oct_gmv
            ),
            100
        ) AS growth_pct

    FROM comparison
)

SELECT
    product_category,

    ROUND(oct_gmv, 2) AS oct_gmv,
    ROUND(nov_gmv, 2) AS nov_gmv,
    ROUND(gmv_increase, 2) AS gmv_increase,
    ROUND(growth_pct, 2) AS growth_pct,

    oct_orders,
    nov_orders,

    ROUND(
        100 * SAFE_DIVIDE(
            gmv_increase,
            SUM(gmv_increase) OVER ()
        ),
        2
    ) AS contribution_to_spike_pct

FROM uplift

WHERE gmv_increase > 0

ORDER BY gmv_increase DESC
"""

category_spike = pd.read_sql(
    text(category_spike_sql),
    con=engine,
)

category_spike.head(10)

In [ ]:
top10 = (
    category_spike
    .head(10)
    .sort_values(
        "gmv_increase",
        ascending=True,
    )
)

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    top10["product_category"],
    top10["gmv_increase"],
)

ax.set_title(
    "Top 10 Categories Contributing to the Nov 2017 GMV Increase"
)

ax.set_xlabel(
    "Incremental GMV vs Oct 2017"
)

ax.set_ylabel(
    "Product Category"
)

ax.grid(
    axis="x",
    alpha=0.2,
)

for bar, pct in zip(
    bars,
    top10["contribution_to_spike_pct"],
):
    ax.annotate(
        f"{pct:.2f}%",
        xy=(
            bar.get_width(),
            bar.get_y() + bar.get_height() / 2,
        ),
        xytext=(3, 0),
        textcoords="offset points",
        va="center",
    )

plt.tight_layout()
plt.show()

### Category contribution finding

The November increase is broad-based rather than being caused by a single category.

The largest contributors observed are:

- bed_bath_table
- health_beauty
- furniture_decor
- watches_gifts
- toys
- computers_accessories

The top six categories account for roughly half of the incremental GMV, which supports the interpretation of a broad shopping-event surge.

## 4. RFM Customer Analysis

RFM analysis evaluates customers using:

- **Recency** — how recently the customer purchased
- **Frequency** — how often the customer purchased
- **Monetary value** — how much the customer spent

The mart table `fct_customer_rfm` contains the precomputed scores and customer segments.

In [ ]:
rfm_top = pd.read_csv(
    OUTPUT_DIR / "rfm_analysis.csv"
)

rfm_segments = pd.read_csv(
    OUTPUT_DIR / "rfm_segment_summary.csv"
)

rfm_segments

In [ ]:
display(
    Image(
        filename=str(
            OUTPUT_DIR / "rfm_segment_summary.png"
        )
    )
)

display(
    Image(
        filename=str(
            OUTPUT_DIR / "rfm_top_customers.png"
        )
    )
)

### RFM observations

Revenue is concentrated among a few major segments, particularly **Big Spenders**, **Needs Attention**, **Lost Customers**, and **Recent One-Time Customers**.

This indicates two important opportunities:

1. protect high-value customers,
2. improve conversion of one-time or declining customers into repeat purchasers.

The top-customer chart also highlights individual high-value customers whose monetary value is substantially above the rest of the customer base.

## 5. Holiday vs Non-Holiday Analysis

The holiday analysis compares commercial activity on dates marked as `TRUE` or `FALSE` in `dim_date.is_holiday`.

`NA` holiday classifications are excluded.

For the current analysis, the period is **Jan 2017 – Jun 2018**.

In [ ]:
holiday = pd.read_csv(
    OUTPUT_DIR / "holiday_impact.csv"
)

holiday_qtr = pd.read_csv(
    OUTPUT_DIR / "holiday_impact_by_quarter.csv"
)

holiday_daily = pd.read_csv(
    OUTPUT_DIR / "holiday_daily_metrics_by_quarter.csv"
)

holiday_category = pd.read_csv(
    OUTPUT_DIR / "holiday_product_category_mix.csv"
)

holiday

### 5.1 Total GMV

In [ ]:
display(
    Image(
        filename=str(
            OUTPUT_DIR / "holiday_impact.png"
        )
    )
)

Total GMV is much larger on non-holidays, but this metric alone is not a fair comparison because there are many more non-holiday calendar days.

For that reason, the normalized measures below are more useful.

### 5.2 Average Orders per Day and Average GMV per Day

In [ ]:
display(
    Image(
        filename=str(
            OUTPUT_DIR / "holiday_avg_orders_per_day.png"
        )
    )
)

display(
    Image(
        filename=str(
            OUTPUT_DIR / "holiday_avg_gmv_per_day.png"
        )
    )
)

### Normalized holiday finding

Across the observed quarters, holiday days generally show:

- fewer orders per day,
- lower GMV per day.

This confirms that the lower holiday GMV is not explained only by the smaller number of holiday dates. Average daily demand is also generally lower on holidays.

### 5.3 Average Order Value by Quarter

In [ ]:
display(
    Image(
        filename=str(
            OUTPUT_DIR / "holiday_aov_by_quarter.png"
        )
    )
)

Holiday AOV is not consistently higher or lower across quarters.

This means the main holiday effect appears to be related more to **order volume** than to a consistent change in the amount customers spend per transaction.

### 5.4 Product Category Mix

In [ ]:
display(
    Image(
        filename=str(
            OUTPUT_DIR / "holiday_product_category_mix.png"
        )
    )
)

### Holiday product-mix finding

Although holiday days tend to have lower overall activity, the mix of products changes.

Categories such as `watches_gifts` and `cool_stuff` have a higher share of holiday GMV, while some categories such as `computers_accessories` contribute a larger share on non-holiday days.

Therefore, holidays appear to affect **what customers buy** more clearly than they increase total business activity.

## 6. Key Business Findings

### Regional sales

- Southeast Brazil is the dominant GMV region.
- South and Northeast form the second tier.
- November 2017 shows a nationwide sales spike.

### November 2017 spike

- The spike is primarily **order-volume driven**.
- AOV did not increase with GMV.
- 24 Nov 2017 is the dominant daily sales peak.
- The increase is broad-based across multiple product categories.
- The top categories contributing to incremental GMV include bed/bath/table, health/beauty, furniture/decor, watches/gifts, toys and computer accessories.

### RFM

- Revenue is concentrated in Big Spenders, Needs Attention, Lost Customers and Recent One-Time Customers.
- The customer base presents a meaningful retention and reactivation opportunity.
- A small number of individual customers have exceptionally high monetary value.

### Holiday behaviour

- Non-holiday days generally generate more orders and GMV per day.
- Holiday AOV varies by quarter and does not show a consistent uplift.
- Holidays change the product-category mix, with some gift-oriented categories gaining relative share.

## 7. Conclusion

The analysis demonstrates how the Olist warehouse can be used to move from descriptive reporting to business investigation.

The regional trend identified a November 2017 anomaly. Drill-down analysis then showed that the increase was driven mainly by higher transaction volume, concentrated around a short event period and spread across multiple product categories.

RFM analysis highlights customer-value concentration and retention opportunities, while the holiday analysis shows that holidays affect product mix more strongly than overall daily demand.

Together, these analyses demonstrate the value of the star-schema warehouse for reusable, scalable and business-oriented analytics.